# 01 — Data Collection & Aggregation — Home Credit Default Risk

**Goal of this notebook**

Build a single, clean, analytical table at the `SK_ID_CURR` (one row per applicant) level by joining
and aggregating the seven raw Home Credit tables, while explicitly guarding against:

1. **Target / temporal leakage** — accidentally using information about the *current* loan that would
   not be available at the time of scoring, or having a bug that leaks `TARGET` into engineered features.
2. **Row duplication ("fan-out")** — child tables have a many-to-one relationship with the applicant
   (a client can have many previous loans, each with many monthly balance/installment rows). If a merge
   is done incorrectly, `application_train` will end up with more rows than applicants, silently corrupting
   every downstream model.

**Table relationships**

```
application_train (1 row per SK_ID_CURR)
│
├── bureau (many rows per SK_ID_CURR)                       [key: SK_ID_CURR]
│      └── bureau_balance (many rows per SK_ID_BUREAU)      [key: SK_ID_BUREAU]
│
└── previous_application (many rows per SK_ID_CURR)         [key: SK_ID_CURR, SK_ID_PREV]
       ├── POS_CASH_balance (many rows per SK_ID_PREV)      [key: SK_ID_PREV]
       ├── credit_card_balance (many rows per SK_ID_PREV)   [key: SK_ID_PREV]
       └── installments_payments (many rows per SK_ID_PREV) [key: SK_ID_PREV]
```

**Aggregation strategy (bottom-up)**

1. Aggregate `bureau_balance` → one row per `SK_ID_BUREAU`, merge into `bureau`.
2. Aggregate enriched `bureau` → one row per `SK_ID_CURR`.
3. Aggregate `POS_CASH_balance`, `credit_card_balance`, `installments_payments` → one row per `SK_ID_PREV` each.
4. Merge those three into `previous_application`, then aggregate → one row per `SK_ID_CURR`.
5. Left-join both `SK_ID_CURR`-level aggregate tables onto `application_train`.

At every step we **assert** the aggregation produced a unique key and that merges did not change row counts,
so leakage/duplication bugs fail loudly instead of silently corrupting the dataset.


In [1]:
import pandas as pd
import numpy as np
import os
import gc

pd.set_option("display.max_columns", 60)

# Folder holding the raw Home Credit CSV files (adjust if your files live elsewhere)
DATA_DIR = r"C:\Users\Kirolos George\OneDrive - Alexandria University\Desktop\projects\tech_trek_internship\credit risk project structure Project\project-name\data\raw\home-credit-default-risk"


## 1. Load raw tables

We load all seven files and print their shapes as a first sanity check.

In [2]:
FILES = {
    "application_train": "application_train.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
}

raw = {}
for name, fname in FILES.items():
    path = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(path)
    # Downcast float64 -> float32 right after loading: halves memory for every numeric
    # column with no meaningful precision loss for feature engineering purposes.
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype("float32")
    raw[name] = df
    print(f"{name:<24} shape={raw[name].shape}")

application_train    = raw["application_train"]
bureau                = raw["bureau"]
bureau_balance        = raw["bureau_balance"]
previous_application  = raw["previous_application"]
pos_cash              = raw["POS_CASH_balance"]
credit_card           = raw["credit_card_balance"]
installments          = raw["installments_payments"]


application_train        shape=(307511, 122)
bureau                   shape=(1716428, 17)
bureau_balance           shape=(27299925, 3)
previous_application     shape=(1670214, 37)
POS_CASH_balance         shape=(10001358, 8)
credit_card_balance      shape=(3840312, 23)
installments_payments    shape=(13605401, 8)


## 2. Schema discovery — primary keys & table relationships

Before writing any join, we confirm the schema programmatically instead of assuming it: which column
is the primary key of each table, and how tables relate to each other. This also feeds directly into
the project's **data dictionary / SQL schema** deliverable.

**Primary-key candidate detection**: a column is a PK candidate if it has zero missing values and is
unique across all rows. We restrict this check to integer-typed columns — a continuous numeric column
(an amount, a rate...) can coincidentally be all-unique in a sample without being a real identifier,
which would otherwise produce misleading candidates.

Tables with no single-column PK candidate (`bureau_balance`, `POS_CASH_balance`, `credit_card_balance`,
`installments_payments`) are expected: they are transactional/monthly-log tables with many rows per
entity, which is exactly why we aggregate them before merging.

In [3]:
table_info = {}
for name, df in raw.items():
    table_info[name] = {"rows": len(df), "columns": df.columns.tolist()}

# Primary-key candidates: no missing values, fully unique, restricted to integer-typed columns
primary_keys = {}
for name, df in raw.items():
    candidates = []
    int_like_cols = df.select_dtypes(include=["int64", "int32", "Int64"]).columns
    for column in int_like_cols:
        missing = df[column].isna().sum()
        unique = df[column].nunique()
        if missing == 0 and unique == len(df):
            candidates.append(column)
    primary_keys[name] = candidates

print("--- Primary key candidates ---")
for table, keys in primary_keys.items():
    print(f"\n{table}")
    if keys:
        for key in keys:
            print(f"  PK candidate -> {key}")
    else:
        print("  No single-column PK candidate (transactional / many-rows-per-entity table)")


--- Primary key candidates ---

application_train
  PK candidate -> SK_ID_CURR

bureau
  PK candidate -> SK_ID_BUREAU

bureau_balance
  No single-column PK candidate (transactional / many-rows-per-entity table)

previous_application
  PK candidate -> SK_ID_PREV

POS_CASH_balance
  No single-column PK candidate (transactional / many-rows-per-entity table)

credit_card_balance
  No single-column PK candidate (transactional / many-rows-per-entity table)

installments_payments
  No single-column PK candidate (transactional / many-rows-per-entity table)


**Important real-world caveat about this dataset**: `bureau` and `previous_application` (and everything
under them) are **not** scoped to `application_train` only — Home Credit built them from the full
client population (train *and* test), so a meaningful share of the `SK_ID_CURR` values in `bureau` /
`previous_application` will not be found in `application_train`. That means:

- `bureau -> bureau_balance` and `previous_application -> {POS_CASH_balance, credit_card_balance,
  installments_payments}` are **expected to be self-contained** (child IDs drawn from the parent,
  regardless of train/test), so ratios should be close to 100%. In practice the raw Home Credit files
  have a small number of genuine **orphan rows** — child records whose ID does not exist in the parent
  table at all. This is a known characteristic of the published dataset itself, not a loading bug, and
  it does not cause duplication or leakage: every merge below is a **left join from the parent's side**,
  so an orphan child row is simply left out, not merged incorrectly. It does mean a small amount of
  historical detail is unrecoverable for those specific IDs — worth reporting as a data-quality
  observation rather than treating as a pipeline error.
- `application_train -> bureau` and `application_train -> previous_application` are **expected to be
  partial** (often 80-90%, not 95%+), simply because `application_train` is a subset of the full client
  population these two tables were built from. A partial ratio here is normal, not a bug.

We report exact ratios and orphan counts for all six relationships below. Only a near-zero ratio (a
genuinely broken join, not a handful of orphan rows) should be treated as a hard failure.

In [4]:
relationships = []
key_columns = ["SK_ID_CURR", "SK_ID_BUREAU", "SK_ID_PREV"]

for parent_table, parent_keys in primary_keys.items():
    for parent_key in parent_keys:
        if parent_key not in key_columns:
            continue
        parent_values = set(raw[parent_table][parent_key].dropna().unique())

        for child_table, child_df in raw.items():
            if child_table == parent_table or parent_key not in child_df.columns:
                continue
            child_values = set(child_df[parent_key].dropna().unique())
            if not child_values:
                continue
            match_ratio = len(child_values.intersection(parent_values)) / len(child_values)
            relationships.append({
                "parent_table": parent_table, "parent_key": parent_key,
                "child_table": child_table, "child_key": parent_key,
                "match_ratio": round(match_ratio, 4),
            })

relationships_df = pd.DataFrame(relationships)

# Relationships where the child table's IDs are fully drawn from the parent (expect ~1.0)
self_contained_relationships = [
    ("bureau", "SK_ID_BUREAU", "bureau_balance", "SK_ID_BUREAU"),
    ("previous_application", "SK_ID_PREV", "POS_CASH_balance", "SK_ID_PREV"),
    ("previous_application", "SK_ID_PREV", "credit_card_balance", "SK_ID_PREV"),
    ("previous_application", "SK_ID_PREV", "installments_payments", "SK_ID_PREV"),
]

# Relationships scoped by application_train being a subset of the full client population
# (partial overlap is expected here, not a bug)
application_scoped_relationships = [
    ("application_train", "SK_ID_CURR", "bureau", "SK_ID_CURR"),
    ("application_train", "SK_ID_CURR", "previous_application", "SK_ID_CURR"),
]

all_expected = self_contained_relationships + application_scoped_relationships

relationship_map = relationships_df[
    relationships_df[["parent_table", "parent_key", "child_table", "child_key"]]
    .apply(tuple, axis=1)
    .isin(all_expected)
].copy()
relationship_map["relationship"] = "1 -> N"
relationship_map["note"] = relationship_map[["parent_table", "parent_key", "child_table", "child_key"]].apply(
    lambda r: "self-contained (expect ~1.0)"
    if tuple(r) in self_contained_relationships
    else "train-subset of full population (partial overlap expected)",
    axis=1,
)
relationship_map = relationship_map[
    ["parent_table", "parent_key", "child_table", "child_key", "relationship", "match_ratio", "note"]
].drop_duplicates()

print("--- Relationship map (all six documented joins, with real match ratios) ---")
print(relationship_map.to_string(index=False))

# Sanity check: every relationship must at least be DETECTED (non-trivial overlap).
# A near-zero ratio would mean the join key is wrong / tables are unrelated - a real bug.
# A ratio that is high but not 100% (e.g. 0.85-0.99) is a normal data-quality characteristic
# of the raw Home Credit files (orphan child rows, or application_train being a population subset)
# and is reported, not treated as a failure.
all_found = relationship_map[
    relationship_map[["parent_table", "parent_key", "child_table", "child_key"]]
    .apply(tuple, axis=1)
    .isin(all_expected)
]
assert len(all_found) == len(all_expected), (
    "One or more of the six documented relationships were not detected at all -> "
    "check for a renamed ID column or an empty table."
)
MIN_SANITY_RATIO = 0.5  # below this, it's not "some orphan rows", it's a broken join
assert (all_found["match_ratio"] >= MIN_SANITY_RATIO).all(), (
    f"A relationship has a match ratio below {MIN_SANITY_RATIO:.0%} -> "
    "this looks like a broken join (wrong key, wrong file, or unrelated tables), not ordinary orphan rows."
)
print(f"\n[OK] All six relationships detected with match ratio >= {MIN_SANITY_RATIO:.0%} (no broken joins).")

# Quantify orphan rows for the self-contained relationships specifically - this is the
# data-quality observation worth carrying into the report's "Common Risks / Quality Checks" section.
print("\n--- Orphan-row report for self-contained relationships (child rows with no parent match) ---")
for parent_table, parent_key, child_table, child_key in self_contained_relationships:
    ratio = relationship_map.loc[
        (relationship_map.parent_table == parent_table) & (relationship_map.child_table == child_table),
        "match_ratio",
    ].iloc[0]
    orphan_pct = (1 - ratio) * 100
    flag = "  <-- notable" if orphan_pct > 3 else ""
    print(f"{child_table:<22} -> {parent_table:<20} : {orphan_pct:5.2f}% orphan rows{flag}")
print(
    "\nThese orphan rows are silently excluded by the left-joins below (no duplication, no leakage) - "
    "they simply mean a small amount of historical detail can't be traced back to a known loan/client. "
    "Document this percentage in the EDA / final report as a known data-quality characteristic of the "
    "raw files."
)

# Informational: application_train-linked overlap, printed for context only.
application_scoped_found = relationship_map[
    relationship_map[["parent_table", "parent_key", "child_table", "child_key"]]
    .apply(tuple, axis=1)
    .isin(application_scoped_relationships)
]
print("\n--- application_train coverage (expected to be partial, not a bug) ---")
for _, row in application_scoped_found.iterrows():
    print(f"{row.child_table:<22} clients found in application_train: {row.match_ratio:.1%}")


--- Relationship map (all six documented joins, with real match ratios) ---
        parent_table   parent_key           child_table    child_key relationship  match_ratio                                                       note
   application_train   SK_ID_CURR                bureau   SK_ID_CURR       1 -> N       0.8616 train-subset of full population (partial overlap expected)
   application_train   SK_ID_CURR  previous_application   SK_ID_CURR       1 -> N       0.8589 train-subset of full population (partial overlap expected)
              bureau SK_ID_BUREAU        bureau_balance SK_ID_BUREAU       1 -> N       0.9473                               self-contained (expect ~1.0)
previous_application   SK_ID_PREV      POS_CASH_balance   SK_ID_PREV       1 -> N       0.9600                               self-contained (expect ~1.0)
previous_application   SK_ID_PREV   credit_card_balance   SK_ID_PREV       1 -> N       0.8910                               self-contained (expect ~1.0)


## 3. Helper functions

- `assert_unique_key`: confirms a table has exactly one row per grouping key (used after every
  aggregation step — a hard stop if the aggregation is wrong).
- `assert_row_count_preserved`: confirms a left-merge did not fan out (duplicate rows), which is the
  #1 source of silent leakage/duplication bugs in this kind of multi-table pipeline.
- `aggregate_table`: generic groupby aggregation for **raw** (first-level) columns — numeric columns get
  `mean/min/max/sum/std`, categorical columns get `nunique`, and every group also gets a `_COUNT` of
  underlying rows (e.g. number of previous loans, number of monthly statements — often a very
  predictive feature itself).
- `aggregate_two_level`: used only when the table being aggregated already contains columns produced
  by a *previous* `aggregate_table` call (e.g. aggregating `bureau_enriched`, which already has
  `BB_MONTHS_BALANCE_STD` etc. mixed in with bureau's own raw columns). Applying the same 5-stat set
  again to an already-aggregated column produces meaningless, high-missingness "second-order" features
  like `STD_STD` or `MEAN_STD` (the standard deviation of a standard deviation has no real business
  meaning, and is `NaN` whenever the inner group had only one row). `aggregate_two_level` detects
  already-aggregated columns by their `_MEAN/_MIN/_MAX/_SUM/_STD/_COUNT/_NUNIQUE` suffix and summarizes
  them with only `mean` and `max` (average level and worst-case/peak level across a client's history) —
  both are interpretable — while still applying the full 5-stat set to genuinely raw columns.

- **Identifier safety**: any column matching Home Credit's own `SK_ID_*` naming convention (e.g. a stray `SK_ID_BUREAU` sitting inside a table being aggregated to `SK_ID_CURR`) is automatically excluded from both functions - identifiers are dropped, never averaged, summed, or otherwise treated as a measurement. This is a naming-based rule, not a manually maintained column list, so it keeps working if a new ID column shows up.

- **Memory efficiency at scale**: both functions compute every numeric statistic in a single `groupby().agg(dict)` call rather than one call per column-group followed by a `merge` - at Home Credit's real size (1M+ rows, hundreds of columns) the old merge-based approach could exhaust memory during pandas' internal block consolidation. Results are also downcast back to `float32` immediately after aggregating, since `.agg()` always returns `float64` by default regardless of the input dtype.

- **Re-run safety**: every pipeline cell below that frees memory with `del` first checks whether its own output already exists in memory (e.g. `if "final_df" not in globals()`). If it does, the cell just prints a `[SKIP]` message instead of recomputing - this is what makes it safe to accidentally re-run the same cell twice in a row, which would otherwise crash with `NameError` on an input that an earlier run already deleted. Note this only covers re-running the *same* cell: if you go back and re-run an *earlier* cell after a *later* one already consumed its output, restart the kernel and use Run All instead.

In [5]:
import re


def safe_del(*names: str) -> None:
    """Delete variables by name only if they currently exist - used by the guarded pipeline
    cells below so a cleanup step never raises NameError on a variable already freed."""
    g = globals()
    for name in names:
        if name in g:
            del g[name]

def assert_unique_key(df: pd.DataFrame, key: str, label: str) -> None:
    n_rows, n_unique = len(df), df[key].nunique()
    assert n_rows == n_unique, (
        f"[{label}] DUPLICATE KEY DETECTED after aggregation: "
        f"{n_rows} rows vs {n_unique} unique '{key}' values"
    )
    print(f"[OK] {label}: {n_rows} rows, unique on '{key}'")


def assert_row_count_preserved(before: int, after: int, label: str) -> None:
    assert before == after, (
        f"[{label}] ROW COUNT CHANGED after merge ({before} -> {after}) "
        f"-> merge is fanning out rows, check the join keys / aggregation!"
    )
    print(f"[OK] {label}: row count preserved ({before} -> {after})")


# Home Credit's own naming convention: every identifier column is named SK_ID_<something>
# (SK_ID_CURR, SK_ID_PREV, SK_ID_BUREAU, and any future ID column following the same pattern).
# This is a general, naming-based rule - not a manually maintained list - so it keeps working
# even if a table gets a new ID column we haven't seen before.
ID_COLUMN_PATTERN = re.compile(r"^SK_ID_", re.IGNORECASE)


def is_id_column(column_name: str) -> bool:
    return bool(ID_COLUMN_PATTERN.match(column_name))


# Suffixes that mark a column as the OUTPUT of a previous aggregation call, rather than a raw field.
AGGREGATED_SUFFIXES = ("_MEAN", "_MIN", "_MAX", "_SUM", "_STD", "_COUNT", "_NUNIQUE")


def _split_columns(df: pd.DataFrame, group_col: str):
    """Shared column classification used by both aggregation functions below.
    Returns (raw_numeric_cols, already_aggregated_cols, categorical_cols), with the grouping
    key and any other SK_ID_* identifier column automatically excluded from all three -
    identifiers are never treated as measurements.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != group_col and not is_id_column(c)]
    already_aggregated_cols = [c for c in numeric_cols if c.endswith(AGGREGATED_SUFFIXES)]
    raw_numeric_cols = [c for c in numeric_cols if c not in already_aggregated_cols]

    categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    categorical_cols = [c for c in categorical_cols if c != group_col and not is_id_column(c)]

    return raw_numeric_cols, already_aggregated_cols, categorical_cols


def _aggregate_numeric(df: pd.DataFrame, group_col: str, prefix: str,
                        raw_numeric_cols: list, already_aggregated_cols: list,
                        raw_stats, agg_stats):
    """Aggregate ALL numeric columns in a SINGLE groupby().agg(dict) call, instead of one call
    per column-group followed by a merge. This matters a lot at Home Credit's scale
    (1M+ rows, hundreds of columns): each separate groupby+merge round-trip allocates its own
    large intermediate DataFrame, and merging wide frames triggers pandas' internal
    block-consolidation, which briefly needs one big contiguous array - exactly what was running
    out of memory before. A single dict-based .agg() call computes everything in one pass with
    one result frame, no merge required for the numeric part at all.
    """
    if not raw_numeric_cols and not already_aggregated_cols:
        return df[[group_col]].drop_duplicates().reset_index(drop=True)

    agg_dict = {col: list(raw_stats) for col in raw_numeric_cols}
    agg_dict.update({col: list(agg_stats) for col in already_aggregated_cols})

    result = df.groupby(group_col).agg(agg_dict)
    result.columns = [f"{prefix}_{col}_{stat.upper()}" for col, stat in result.columns]
    # Aggregation results are float64 by default regardless of input dtype - downcast back to
    # float32 immediately so memory doesn't creep back up right after we just saved it on load.
    float_cols = result.select_dtypes(include=["float64"]).columns
    result[float_cols] = result[float_cols].astype("float32")
    return result.reset_index()


def _finalize(df: pd.DataFrame, group_col: str, prefix: str, numeric_result: pd.DataFrame,
              categorical_cols: list) -> pd.DataFrame:
    """Attach categorical nunique + row count to the already-computed numeric aggregation."""
    out = numeric_result

    if categorical_cols:
        cat_agg = df.groupby(group_col)[categorical_cols].nunique()
        cat_agg.columns = [f"{prefix}_{col}_NUNIQUE" for col in cat_agg.columns]
        out = out.merge(cat_agg.reset_index(), on=group_col, how="left")

    count_df = df.groupby(group_col).size().reset_index(name=f"{prefix}_COUNT")
    out = out.merge(count_df, on=group_col, how="left")
    return out


def aggregate_table(df: pd.DataFrame, group_col: str, prefix: str,
                     stats=("mean", "min", "max", "sum", "std")) -> pd.DataFrame:
    """Collapse `df` to one row per `group_col` (first aggregation level - all columns are raw).
    Numeric columns -> the given `stats`, computed in a single groupby().agg(dict) call.
    Categorical columns -> nunique. Any SK_ID_* column other than `group_col` is automatically
    dropped, never aggregated as a measurement. Adds a `{prefix}_COUNT` column.
    """
    raw_numeric_cols, _, categorical_cols = _split_columns(df, group_col)
    numeric_result = _aggregate_numeric(df, group_col, prefix, raw_numeric_cols, [], stats, stats)
    return _finalize(df, group_col, prefix, numeric_result, categorical_cols)


def aggregate_two_level(df: pd.DataFrame, group_col: str, prefix: str,
                         raw_stats=("mean", "min", "max", "sum", "std"),
                         agg_stats=("mean", "max")) -> pd.DataFrame:
    """Like `aggregate_table`, but for a table that mixes raw columns with columns already produced
    by an earlier aggregation step (auto-detected by their _MEAN/_MIN/_MAX/_SUM/_STD/_COUNT/_NUNIQUE
    suffix). Raw columns get the full `raw_stats` set. Already-aggregated columns get only `agg_stats`
    (mean = average level across the client's history, max = worst-case/peak level) - this avoids
    meaningless, high-missingness second-order features such as STD_STD or MEAN_STD. As in
    `aggregate_table`, any SK_ID_* identifier column is automatically excluded, and both column
    groups are computed in a single groupby().agg(dict) call for memory efficiency at scale.
    """
    raw_numeric_cols, already_aggregated_cols, categorical_cols = _split_columns(df, group_col)
    numeric_result = _aggregate_numeric(
        df, group_col, prefix, raw_numeric_cols, already_aggregated_cols, raw_stats, agg_stats
    )
    return _finalize(df, group_col, prefix, numeric_result, categorical_cols)


## 4. Step 1 — `bureau_balance` → `bureau`

`bureau_balance` has many monthly-status rows per `SK_ID_BUREAU`. We aggregate it to one row per
`SK_ID_BUREAU`, then left-merge into `bureau`. Row count of `bureau` must stay exactly the same
after this merge — if it grows, we have a duplication bug.

In [6]:
if "bureau_enriched" not in globals():
    bb_agg = aggregate_table(bureau_balance, group_col="SK_ID_BUREAU", prefix="BB")
    assert_unique_key(bb_agg, "SK_ID_BUREAU", "bureau_balance aggregated")

    n_before = len(bureau)
    bureau_enriched = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
    assert_row_count_preserved(n_before, len(bureau_enriched), "bureau + bureau_balance merge")

    # Consistent with the orphan-row report in Section 2: bb_agg rows whose SK_ID_BUREAU has no match
    # in bureau are simply not merged in (left join from bureau's side) - report how many were skipped.
    orphan_bb = bb_agg.loc[~bb_agg["SK_ID_BUREAU"].isin(bureau["SK_ID_BUREAU"])]
    print(f"bureau_balance groups skipped (no matching SK_ID_BUREAU in bureau): {len(orphan_bb)} "
          f"({len(orphan_bb) / len(bb_agg):.2%} of aggregated bureau_balance groups)")

    # bureau_balance itself (millions of rows) is no longer needed once bb_agg is computed - free it.
    del bb_agg
    safe_del("bureau_balance")
    gc.collect()
else:
    print("[SKIP] bureau_enriched already computed in this session - not recomputing.")


[OK] bureau_balance aggregated: 817395 rows, unique on 'SK_ID_BUREAU'
[OK] bureau + bureau_balance merge: row count preserved (1716428 -> 1716428)
bureau_balance groups skipped (no matching SK_ID_BUREAU in bureau): 43041 (5.27% of aggregated bureau_balance groups)


## 5. Step 2 — `bureau` (+ `bureau_balance`) → `SK_ID_CURR` level

Now collapse the enriched bureau table (many previous credits per client) down to one row per applicant.

**Note**: `bureau_enriched` already contains `BB_*` columns produced by aggregating `bureau_balance` above, mixed in with bureau's own raw columns. We use `aggregate_two_level` here (instead of `aggregate_table`) so those already-aggregated columns are summarized with only `mean`/`max`, instead of re-applying the full 5-stat set and producing meaningless combinations like `BB_MONTHS_BALANCE_STD_STD`.

In [7]:
bureau_agg = aggregate_two_level(bureau_enriched, group_col="SK_ID_CURR", prefix="BUREAU")
assert_unique_key(bureau_agg, "SK_ID_CURR", "bureau aggregated to SK_ID_CURR")


[OK] bureau aggregated to SK_ID_CURR: 305811 rows, unique on 'SK_ID_CURR'


## 6. Step 3 — `POS_CASH_balance`, `credit_card_balance`, `installments_payments` → `SK_ID_PREV` level

These three tables describe the monthly life of a *previous* loan (`SK_ID_PREV`). Each also carries a redundant `SK_ID_CURR` column; `aggregate_table` detects and drops any `SK_ID_*` column automatically (no manual `.drop(columns=...)` needed), and collapses each table to one row per `SK_ID_PREV`.

In [8]:
if "pos_agg" not in globals():
    # No manual column dropping needed: aggregate_table auto-excludes the stray SK_ID_CURR
    # column in each of these tables (it is not the grouping key here, SK_ID_PREV is).
    pos_agg  = aggregate_table(pos_cash, group_col="SK_ID_PREV", prefix="POS")
    cc_agg   = aggregate_table(credit_card, group_col="SK_ID_PREV", prefix="CC")
    inst_agg = aggregate_table(installments, group_col="SK_ID_PREV", prefix="INST")

    assert_unique_key(pos_agg, "SK_ID_PREV", "POS_CASH_balance aggregated")
    assert_unique_key(cc_agg, "SK_ID_PREV", "credit_card_balance aggregated")
    assert_unique_key(inst_agg, "SK_ID_PREV", "installments_payments aggregated")

    # These raw tables (installments_payments alone is ~13.6M rows in the real dataset) are no
    # longer needed once aggregated to SK_ID_PREV level - free them before the next merge step.
    safe_del("pos_cash", "credit_card", "installments")
    gc.collect()
else:
    print("[SKIP] pos_agg/cc_agg/inst_agg already computed in this session - not recomputing.")


[OK] POS_CASH_balance aggregated: 936325 rows, unique on 'SK_ID_PREV'
[OK] credit_card_balance aggregated: 104307 rows, unique on 'SK_ID_PREV'
[OK] installments_payments aggregated: 997752 rows, unique on 'SK_ID_PREV'


## 7. Step 4 — merge child aggregates into `previous_application`, then collapse to `SK_ID_CURR`

Each of the three aggregates above is joined onto `previous_application` on `SK_ID_PREV` — this must
**not** change the row count of `previous_application` (each `SK_ID_PREV` is unique there and in the
aggregates, so it's a clean 1-to-1 join). We then aggregate the enriched table down to one row per
applicant, exactly like we did for bureau.

**Note**: `prev_enriched` mixes previous_application's own raw columns with already-aggregated `POS_*`, `CC_*`, `INST_*` columns from the step above. We use `aggregate_two_level` here for the same reason as Step 2 - to avoid meaningless second-order features on the already-aggregated columns.

In [9]:
if "prev_agg" not in globals():
    n_before = len(previous_application)
    prev_enriched = (
        previous_application
        .merge(pos_agg, on="SK_ID_PREV", how="left")
        .merge(cc_agg, on="SK_ID_PREV", how="left")
        .merge(inst_agg, on="SK_ID_PREV", how="left")
    )
    assert_row_count_preserved(n_before, len(prev_enriched), "previous_application + POS/CC/INST merge")

    # Same orphan-row accounting as for bureau/bureau_balance above.
    for label, agg_df in [("POS_CASH_balance", pos_agg), ("credit_card_balance", cc_agg), ("installments_payments", inst_agg)]:
        orphan = agg_df.loc[~agg_df["SK_ID_PREV"].isin(previous_application["SK_ID_PREV"])]
        print(f"{label} groups skipped (no matching SK_ID_PREV in previous_application): {len(orphan)} "
              f"({len(orphan) / len(agg_df):.2%})")

    prev_agg = aggregate_two_level(prev_enriched, group_col="SK_ID_CURR", prefix="PREV")
    assert_unique_key(prev_agg, "SK_ID_CURR", "previous_application aggregated to SK_ID_CURR")

    # prev_enriched (1.67M+ rows with all POS/CC/INST columns merged in) is no longer needed
    # once aggregated to SK_ID_CURR level - free it before the final application_train merge.
    safe_del("prev_enriched", "pos_agg", "cc_agg", "inst_agg", "previous_application")
    gc.collect()
else:
    print("[SKIP] prev_agg already computed in this session - not recomputing.")


[OK] previous_application + POS/CC/INST merge: row count preserved (1670214 -> 1670214)
POS_CASH_balance groups skipped (no matching SK_ID_PREV in previous_application): 37422 (4.00%)
credit_card_balance groups skipped (no matching SK_ID_PREV in previous_application): 11372 (10.90%)
installments_payments groups skipped (no matching SK_ID_PREV in previous_application): 38847 (3.89%)
[OK] previous_application aggregated to SK_ID_CURR: 338857 rows, unique on 'SK_ID_CURR'


## 8. Step 5 — final merge into `application_train`

This is the highest-risk step for silent duplication: both `bureau_agg` and `prev_agg` are guaranteed
unique on `SK_ID_CURR` (asserted above), so a left-merge from `application_train` **must** preserve its
row count exactly. We assert this explicitly rather than trusting it.

### Leakage note
`bureau` and `previous_application` (and their children) describe loans that are **separate from, and
prior to, the current application** — they are legitimate predictive history, not leakage. The
practical leakage risks in this pipeline are (a) accidentally duplicating rows so the model implicitly
sees the target multiple times, and (b) a coding bug that merges in a column derived from `TARGET`. We
check for both below.

In [10]:
if "final_df" not in globals():
    n_before = len(application_train)
    final_df = (
        application_train
        .merge(bureau_agg, on="SK_ID_CURR", how="left")
        .merge(prev_agg, on="SK_ID_CURR", how="left")
    )
    assert_row_count_preserved(n_before, len(final_df), "application_train final merge")

    # Free everything that only existed to produce bureau_agg / prev_agg - final_df now has
    # everything we need going forward.
    safe_del("bureau_enriched", "bureau_agg", "prev_agg", "bureau", "application_train")
    gc.collect()
else:
    print("[SKIP] final_df already computed in this session - not recomputing.")


[OK] application_train final merge: row count preserved (307511 -> 307511)


## 9. Final sanity, duplication & leakage checks

Before handing this off to EDA, we run one consolidated set of checks:

1. No fully duplicated rows.
2. No duplicated `SK_ID_CURR` (one row per applicant, guaranteed).
3. `TARGET` has no missing values (every training row must have a label).
4. A leakage "smell test": flag any feature with |correlation| > 0.95 with `TARGET` for manual review —
   a real predictive feature is very unlikely to be that strongly correlated; it usually signals a bug.
5. Report missingness introduced by the left-joins (applicants with no bureau/previous-loan history will
   have NaNs in those new columns — this is expected and should be handled explicitly in EDA/feature
   engineering, e.g. with a `HAS_BUREAU_HISTORY` flag, not silently dropped).

In [11]:
print("Final merged shape:", final_df.shape)

# 1. Fully duplicated rows
n_dupe_rows = final_df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupe_rows}")
assert n_dupe_rows == 0, "Found fully duplicated rows in the final table!"

# 2. Duplicated SK_ID_CURR
n_dupe_ids = final_df["SK_ID_CURR"].duplicated().sum()
print(f"Duplicated SK_ID_CURR: {n_dupe_ids}")
assert n_dupe_ids == 0, "Found duplicated SK_ID_CURR in the final table!"

# 3. TARGET completeness
n_missing_target = final_df["TARGET"].isnull().sum()
print(f"Missing TARGET values: {n_missing_target}")
assert n_missing_target == 0, "TARGET has missing values after merging!"

# 4. Leakage smell test
numeric_final = final_df.select_dtypes(include=[np.number])
corrs = numeric_final.corr()["TARGET"].drop("TARGET").abs().sort_values(ascending=False)
suspicious = corrs[corrs > 0.95]
print(f"\nFeatures with |corr| > 0.95 with TARGET (review for leakage): {len(suspicious)}")
if len(suspicious):
    print(suspicious)

# 5. Missingness introduced by the joins
new_cols = [c for c in final_df.columns if c.startswith("BUREAU_") or c.startswith("PREV_")]
missing_pct = final_df[new_cols].isnull().mean().sort_values(ascending=False)
print(f"\nTop 10 engineered columns by % missing (expected for applicants with no history):")
print((missing_pct.head(10) * 100).round(1).astype(str) + "%")

print("\nAll checks passed - dataset is ready for EDA.")


Final merged shape: (307511, 629)
Fully duplicated rows: 0
Duplicated SK_ID_CURR: 0
Missing TARGET values: 0

Features with |corr| > 0.95 with TARGET (review for leakage): 0

Top 10 engineered columns by % missing (expected for applicants with no history):
PREV_RATE_INTEREST_PRIVILEGED_STD       99.9%
PREV_RATE_INTEREST_PRIMARY_STD          99.9%
PREV_RATE_INTEREST_PRIVILEGED_MEAN      98.5%
PREV_RATE_INTEREST_PRIMARY_MAX          98.5%
PREV_RATE_INTEREST_PRIMARY_MIN          98.5%
PREV_RATE_INTEREST_PRIVILEGED_MAX       98.5%
PREV_RATE_INTEREST_PRIVILEGED_MIN       98.5%
PREV_RATE_INTEREST_PRIMARY_MEAN         98.5%
PREV_CC_AMT_PAYMENT_CURRENT_STD_MEAN    83.0%
PREV_CC_AMT_PAYMENT_CURRENT_STD_MAX     83.0%
dtype: object

All checks passed - dataset is ready for EDA.


## 10. Preview the collected dataset

This is the single analytical table the EDA notebook will start from — one row per applicant,
combining application data with aggregated bureau and previous-application history.

In [12]:
final_df.head(5)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,PREV_INST_AMT_INSTALMENT_STD_MAX,PREV_INST_AMT_PAYMENT_MEAN_MEAN,PREV_INST_AMT_PAYMENT_MEAN_MAX,PREV_INST_AMT_PAYMENT_MIN_MEAN,PREV_INST_AMT_PAYMENT_MIN_MAX,PREV_INST_AMT_PAYMENT_MAX_MEAN,PREV_INST_AMT_PAYMENT_MAX_MAX,PREV_INST_AMT_PAYMENT_SUM_MEAN,PREV_INST_AMT_PAYMENT_SUM_MAX,PREV_INST_AMT_PAYMENT_STD_MEAN,PREV_INST_AMT_PAYMENT_STD_MAX,PREV_INST_COUNT_MEAN,PREV_INST_COUNT_MAX,PREV_NAME_CONTRACT_TYPE_NUNIQUE,PREV_WEEKDAY_APPR_PROCESS_START_NUNIQUE,PREV_FLAG_LAST_APPL_PER_CONTRACT_NUNIQUE,PREV_NAME_CASH_LOAN_PURPOSE_NUNIQUE,PREV_NAME_CONTRACT_STATUS_NUNIQUE,PREV_NAME_PAYMENT_TYPE_NUNIQUE,PREV_CODE_REJECT_REASON_NUNIQUE,PREV_NAME_TYPE_SUITE_NUNIQUE,PREV_NAME_CLIENT_TYPE_NUNIQUE,PREV_NAME_GOODS_CATEGORY_NUNIQUE,PREV_NAME_PORTFOLIO_NUNIQUE,PREV_NAME_PRODUCT_TYPE_NUNIQUE,PREV_CHANNEL_TYPE_NUNIQUE,PREV_NAME_SELLER_INDUSTRY_NUNIQUE,PREV_NAME_YIELD_GROUP_NUNIQUE,PREV_PRODUCT_COMBINATION_NUNIQUE,PREV_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,...,10058.038086,11559.247070,11559.247070,9251.775391,9251.775391,53093.746094,53093.746094,219625.703125,2.196257e+05,10058.038086,10058.038086,19.000000,19.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,174800.390625,78558.484375,164425.343750,56431.859375,98356.992188,210713.453125,560835.375000,539621.562500,1.150977e+06,58313.691406,174800.390625,8.333333,12.0,2.0,3.0,1.0,2.0,1.0,2.0,1.0,2.0,2.0,3.0,2.0,2.0,3.0,3.0,2.0,3.0,3.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,...,3011.871826,7096.154785,7096.154785,5357.250000,5357.250000,10573.964844,10573.964844,21288.464844,2.128846e+04,3011.871826,3011.871826,3.000000,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,...,5669.203613,241944.234375,691786.875000,241099.109375,691786.875000,245324.687500,691786.875000,335717.781250,6.917869e+05,2834.601807,5669.203613,5.333333,10.0,3.0,4.0,1.0,2.0,3.0,2.0,2.0,2.0,2.0,3.0,4.0,2.0,3.0,3.0,4.0,7.0,9.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,-19932,-3038,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,...,112.415260,11671.539062,20919.408203,6785.765625,16046.099609,12132.369141,22678.785156,161225.593750,2.801997e+05,1540.309082,6278.775879,13.200000,17.0,2.0,5.0,1.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,3.0,4.0,3.0,2.0,4.0,6.0


## 11. Save the collected dataset

Persist the merged table so the EDA notebook can load it directly instead of re-running this pipeline.

In [13]:
OUTPUT_DIR = r"C:\Users\Kirolos George\OneDrive - Alexandria University\Desktop\projects\tech_trek_internship\credit risk project structure Project\project-name\src\data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, "collected_dataset.csv")

final_df.to_csv(output_path, index=False)

print(f"Saved collected dataset: {output_path}")
print(f"Shape: {final_df.shape}")


Saved collected dataset: C:\Users\Kirolos George\OneDrive - Alexandria University\Desktop\projects\tech_trek_internship\credit risk project structure Project\project-name\src\data\collected_dataset.csv
Shape: (307511, 629)
